# Run all

Runs the whole weekly update in order: the ESPN weekly pull, every FantasyLuck notebook (01 through the export), then `01_build_data` and `02_build_site`. Each notebook runs in its own folder, the same as opening it and choosing Run All. If one fails, this stops there and shows which notebook and cell.

GitHub runs this every Wednesday. You can also run it yourself the same way.

In [ ]:
from pathlib import Path
import time
import nbformat
from nbclient import NotebookClient
from nbclient.exceptions import CellExecutionError

WFR = Path(".").resolve()
STEPS = ([WFR / "League" / "00_espn_weekly.ipynb"]
         + sorted((WFR / "FantasyLuck").glob("[0-9][0-9]_*.ipynb"))     # 01, 02, ... in order, ending with the export
         + [WFR / "League" / "01_build_data.ipynb", WFR / "League" / "02_build_site.ipynb"])
for s in STEPS:
    print(s.relative_to(WFR), "" if s.exists() else "  ← missing")

## Run

Notebooks run without saving their outputs, so the files stay clean between weeks.

In [ ]:
start = time.time()
for nb_path in STEPS:
    t = time.time()
    nb = nbformat.read(nb_path, as_version=4)
    client = NotebookClient(nb, timeout=3600, kernel_name="python3", resources={"metadata": {"path": str(nb_path.parent)}})
    try:
        client.execute()
    except CellExecutionError as e:
        print(f"Stopped: {nb_path.relative_to(WFR)} failed.\n")
        raise
    print(f"{nb_path.relative_to(WFR)}  ({time.time() - t:.0f}s)")
print(f"\nDone in {(time.time() - start) / 60:.1f} minutes. The site is in {WFR / 'site'}")